# R.O.A.D. Barbados — Colab bootstrap

Run cells top to bottom at the start of every session — the Colab VM is ephemeral, so all of this (rclone install, deps, data/model pull) has to happen fresh each time. Code and data live on Google account **B**'s Drive; this notebook runs under whichever account (**A**) currently has a Colab runtime. See `runners/colab/README.md` for the one-time `rclone.conf` / Secrets setup this depends on — do that first if you haven't.

Do not print `userdata.get('RCLONE_CONF')` anywhere, and never commit `rclone.conf` to git.

## 1. Check GPU tier

In [ ]:
import subprocess

gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu_info.returncode != 0 or not gpu_info.stdout.strip():
    raise RuntimeError("No GPU detected — Runtime > Change runtime type > GPU, then re-run this cell.")

gpu_name, gpu_mem = [x.strip() for x in gpu_info.stdout.strip().splitlines()[0].split(",")]
print(f"GPU: {gpu_name}   VRAM: {gpu_mem}")

TIER_HINTS = {
    "T4": "small — use the Qwen2.5-VL 2B/3B backbone, 4-bit QLoRA, small batches",
    "L4": "mid — 3B/7B backbone should fit with 4-bit QLoRA",
    "A100": "large — 7B backbone comfortable, can try higher precision",
    "V100": "mid — treat like T4/L4, 4-bit QLoRA recommended",
    "P100": "small/older — 2B backbone, 4-bit QLoRA",
}
hint = next((v for k, v in TIER_HINTS.items() if k in gpu_name), "unknown tier — check VRAM above and size the backbone accordingly")
print(f"Recommendation: {hint}")
print("Free-tier Colab does not guarantee a specific GPU — Pro/Pro+ improves odds of A100/L4 over T4.")

## 2. Install rclone (fresh each session — the VM is ephemeral)

In [ ]:
!apt-get -qq update && apt-get -qq install -y rclone
!rclone version | head -1

## 3. Restore `rclone.conf` from the Colab secret

Requires a secret named `RCLONE_CONF` set once via the key icon in the left sidebar, with "Notebook access" enabled for this notebook. Setup instructions: `runners/colab/README.md`.

In [ ]:
import os
import base64
import getpass
from pathlib import Path

RCLONE_CONF_SECRET_NAME = "RCLONE_CONF"

conf_dir = Path.home() / ".config" / "rclone"
conf_dir.mkdir(parents=True, exist_ok=True)
conf_path = conf_dir / "rclone.conf"

conf_text = None
try:
    from google.colab import userdata
    conf_text = userdata.get(RCLONE_CONF_SECRET_NAME)
except Exception as exc:
    # Colab Secrets only relay through the colab.research.google.com web
    # frontend — connecting from an external frontend (e.g. VS Code's
    # Jupyter extension attached to a Colab kernel) can't reach them, and
    # userdata.get() times out / raises instead. Fall back to pasting the
    # file in directly, base64-encoded so a multi-line file survives a
    # single getpass() prompt (getpass only reads one line).
    print(f"Colab Secrets unavailable here ({type(exc).__name__}) — falling back to manual entry.")
    print("On your local machine (PowerShell), copy your rclone.conf as one base64 line:")
    print('  [Convert]::ToBase64String([IO.File]::ReadAllBytes("$env:APPDATA\\rclone\\rclone.conf")) | Set-Clipboard')
    b64 = getpass.getpass("Paste it here (hidden input): ")
    conf_text = base64.b64decode(b64).decode("utf-8")
    del b64

conf_path.write_text(conf_text)
os.chmod(conf_path, 0o600)
del conf_text  # never print/log this
print(f"rclone.conf restored to {conf_path} ({conf_path.stat().st_size} bytes)")

## 4. Verify connectivity + define remote/local paths

`REMOTE` must match the remote name you gave `rclone config` when authorizing against Drive B — the convention this whole runner set expects is `roadB`.

In [ ]:
REMOTE = "roadB"
DRIVE_ROOT = f"{REMOTE}:road-barbados-htr"

check = subprocess.run(["rclone", "lsd", f"{REMOTE}:"], capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError(
        "rclone can't reach the 'roadB' remote. Check: (1) the RCLONE_CONF secret is set and "
        "enabled for this notebook, (2) the remote is actually named 'roadB' in that config, "
        "(3) the refresh token hasn't been revoked (re-run `rclone config` against account B if so). "
        "stderr:\n" + check.stderr
    )
print(check.stdout)
print("rclone -> Drive B connectivity OK")

In [ ]:
TIER = "vlm"  # "vlm" or "kraken" — which stack this session runs

CODE_DIR = "/content/road-barbados-htr"
DATA_DIR = "/content/data"
MODEL_CACHE_DIR = "/content/model_cache"
CKPT_DIR = f"/content/checkpoints/{TIER}"
CKPT_REMOTE = f"{DRIVE_ROOT}/experiments/{TIER}"

os.environ["HF_HOME"] = f"{MODEL_CACHE_DIR}/hf"
os.environ["TRANSFORMERS_CACHE"] = os.environ["HF_HOME"]

for d in (CODE_DIR, DATA_DIR, MODEL_CACHE_DIR, CKPT_DIR, os.environ["HF_HOME"]):
    os.makedirs(d, exist_ok=True)

print(f"TIER={TIER}")
print(f"CODE_DIR={CODE_DIR}  DATA_DIR={DATA_DIR}  MODEL_CACHE_DIR={MODEL_CACHE_DIR}  CKPT_DIR={CKPT_DIR}")

## 5. Pull code from Drive

Everything except the heavy dirs (images, model weights, checkpoints), which get pulled separately below straight to the paths training actually reads from.

In [ ]:
!rclone copy "{DRIVE_ROOT}" "{CODE_DIR}" -P \
  --exclude "data/images/**" \
  --exclude "data/*.csv" \
  --exclude "model_cache/**" \
  --exclude "experiments/**" \
  --exclude ".git/**" \
  --exclude ".venv-*/**"

## 6. Install Python deps (fresh each session)

In [ ]:
!pip install -q -r "{CODE_DIR}/requirements.txt"
!pip install -q -r "{CODE_DIR}/src/{TIER}/requirements.txt"

## 7. Pull data to local disk

Reading 5472 small files remotely on every batch would bottleneck the dataloader — always train/infer against `DATA_DIR` on local Colab disk, never against the `roadB:` remote directly.

In [ ]:
!rclone copy "{DRIVE_ROOT}/data/images" "{DATA_DIR}/images" -P
!rclone copy "{DRIVE_ROOT}/data" "{DATA_DIR}" -P --include "*.csv" --max-depth 1

## 8. Pull cached model weights (if any exist from a previous session)

A 7B model shouldn't be re-downloaded from Hugging Face every session. `push_model_cache()` below pushes newly-downloaded weights back up — call it once after `src/vlm/train.py` or `src/vlm/infer.py` finishes loading the backbone for the first time this session.

In [ ]:
!rclone copy "{DRIVE_ROOT}/model_cache" "{MODEL_CACHE_DIR}" -P

def push_model_cache():
    """Call after a fresh HF download so future sessions reuse it instead of re-downloading."""
    subprocess.run(["rclone", "copy", MODEL_CACHE_DIR, f"{DRIVE_ROOT}/model_cache", "-P"], check=True)
    print("model_cache pushed to Drive B")

## 9. Checkpoint detect-and-resume

Pulls any existing checkpoint for this `TIER` down from Drive B. `RESUME_FROM_CHECKPOINT` is `None` on a first run, or a local path to pass straight into `--resume_from_checkpoint` (VLM/HF Trainer) once `src/vlm/train.py` exists.

In [ ]:
import re
from pathlib import Path

!rclone copy "{CKPT_REMOTE}" "{CKPT_DIR}" -P

def find_latest_checkpoint(ckpt_dir):
    ckpt_dir = Path(ckpt_dir)
    candidates = sorted(
        (p for p in ckpt_dir.glob("checkpoint-*") if p.is_dir()),
        key=lambda p: int(re.search(r"\d+$", p.name).group()),
    )
    return str(candidates[-1]) if candidates else None

RESUME_FROM_CHECKPOINT = find_latest_checkpoint(CKPT_DIR)
if RESUME_FROM_CHECKPOINT:
    print(f"Resuming from {RESUME_FROM_CHECKPOINT}")
else:
    print("No existing checkpoint found on Drive B — starting fresh")

## 10. Periodic checkpoint push-back

Colab session/idle timeouts can happen with no warning, especially on the free tier. `RcloneCheckpointSyncCallback` mirrors `CKPT_DIR` to Drive B every time the trainer saves, so at most one save interval's progress is ever at risk. Wire it into `src/vlm/train.py` once that script exists:

```python
Trainer(..., callbacks=[RcloneCheckpointSyncCallback(CKPT_DIR, CKPT_REMOTE)])
```

In [ ]:
import time

try:
    from transformers import TrainerCallback
except ImportError:
    TrainerCallback = object

class RcloneCheckpointSyncCallback(TrainerCallback):
    def __init__(self, local_dir, remote_dir, min_seconds_between_syncs=60):
        self.local_dir = local_dir
        self.remote_dir = remote_dir
        self.min_seconds_between_syncs = min_seconds_between_syncs
        self._last_sync = 0.0

    def _sync(self):
        now = time.time()
        if now - self._last_sync < self.min_seconds_between_syncs:
            return
        subprocess.run(["rclone", "copy", self.local_dir, self.remote_dir, "-P"], check=False)
        self._last_sync = now
        print(f"[sync] checkpoint mirrored to {self.remote_dir}")

    def on_save(self, args, state, control, **kwargs):
        self._sync()
        return control

    def on_train_end(self, args, state, control, **kwargs):
        self._sync()
        return control

print("RcloneCheckpointSyncCallback ready")

## 11. Run training / inference

`src/kraken/train.py`, `src/vlm/train.py`, `src/vlm/infer.py` are built in a later prompt. Once they exist, invoke them against the local paths set up above, e.g.:

```python
!cd "{CODE_DIR}" && python src/vlm/train.py \
    --config src/vlm/config.yaml \
    --train_csv "{DATA_DIR}/Train.csv" \
    --base_image_dir "{DATA_DIR}/images" \
    --output_dir "{CKPT_DIR}" \
    $( [ -n "{RESUME_FROM_CHECKPOINT}" ] && echo --resume_from_checkpoint "{RESUME_FROM_CHECKPOINT}" )
```

After training/inference, remember to `push_model_cache()` if this session downloaded fresh weights, and do a final `!rclone copy "{CKPT_DIR}" "{CKPT_REMOTE}" -P` — the periodic callback covers mid-run saves, but a manual final sync after `trainer.train()` returns is cheap insurance.